<a href="https://colab.research.google.com/github/CamiloVga/IA-Codes/blob/main/Rag_Base_de_Datos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# RAG BASE DE DATOS CON GROQ API (Repo Github)


# ===== INSTALACIÓN =====
!pip install groq sentence-transformers pandas numpy scikit-learn -q

# ===== CÓDIGO RAG MINIMALISTA =====
import pandas as pd
import numpy as np
import sqlite3
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from groq import Groq
from google.colab import userdata

class RAGSimple:
    def __init__(self):
        print("Iniciando RAG...")
        self.model = SentenceTransformer('all-MiniLM-L6-v2')
        self.conn = None
        self.table_name = 'data'
        self.column_embeddings = {}
        self.column_types = {}

    def cargar_datos(self, csv_url):
        """Cargar CSV y crear base de datos"""
        print("Cargando datos...")
        df = pd.read_csv(csv_url)
        self.conn = sqlite3.connect(':memory:')
        df.to_sql(self.table_name, self.conn, index=False, if_exists='replace')

        # Analizar columnas
        for col in df.columns:
            if df[col].dtype in ['int64', 'float64']:
                self.column_types[col] = 'numeric'
                desc = f"{col} valores numéricos entre {df[col].min()} y {df[col].max()}"
            elif df[col].nunique() <= 20:
                self.column_types[col] = 'categorical'
                valores = ', '.join(map(str, df[col].unique()[:5]))
                desc = f"{col} categorías como: {valores}"
            else:
                self.column_types[col] = 'text'
                desc = f"{col} texto libre"

            self.column_embeddings[col] = self.model.encode([desc])[0]

        print(f"✓ Datos cargados: {df.shape[0]} filas, {df.shape[1]} columnas")
        return df.shape

    def encontrar_columnas(self, pregunta):
        """Encontrar columnas relevantes"""
        pregunta_emb = self.model.encode([pregunta])[0]
        scores = []

        for col, col_emb in self.column_embeddings.items():
            sim = cosine_similarity([pregunta_emb], [col_emb])[0][0]
            scores.append((col, sim))

        return sorted(scores, key=lambda x: x[1], reverse=True)[:3]

    def generar_sql(self, pregunta):
        """Generar SQL inteligente"""
        cols_relevantes = [col for col, _ in self.encontrar_columnas(pregunta)]
        cols_numericas = [c for c in cols_relevantes if self.column_types.get(c) == 'numeric']
        cols_categoricas = [c for c in cols_relevantes if self.column_types.get(c) == 'categorical']

        pregunta_lower = pregunta.lower()

        # Detección de intención mejorada
        if any(palabra in pregunta_lower for palabra in ['más vendido', 'más popular', 'qué producto', 'cuáles productos']):
            # Productos más vendidos por cantidad
            if cols_numericas and cols_categoricas:
                return f"""
                SELECT {cols_categoricas[0]} as producto,
                       SUM({cols_numericas[0]}) as total_cantidad,
                       COUNT(*) as num_ventas
                FROM {self.table_name}
                GROUP BY {cols_categoricas[0]}
                ORDER BY total_cantidad DESC
                LIMIT 10
                """

        if any(palabra in pregunta_lower for palabra in ['método pago', 'forma pago', 'cómo pagan', 'pago más común']):
            # Análisis de métodos de pago
            return f"""
            SELECT Método_pago,
                   COUNT(*) as cantidad,
                   ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM {self.table_name}), 1) as porcentaje
            FROM {self.table_name}
            GROUP BY Método_pago
            ORDER BY cantidad DESC
            """

        if any(palabra in pregunta_lower for palabra in ['ingresos', 'revenue', 'dinero', 'ganancias']):
            # Análisis de ingresos
            if 'Precio_unitario' in self.column_types and 'Cantidad' in self.column_types:
                return f"""
                SELECT {cols_categoricas[0] if cols_categoricas else 'Producto'} as categoria,
                       SUM(Precio_unitario * Cantidad) as total_ingresos,
                       AVG(Precio_unitario * Cantidad) as ingreso_promedio
                FROM {self.table_name}
                GROUP BY {cols_categoricas[0] if cols_categoricas else 'Producto'}
                ORDER BY total_ingresos DESC
                LIMIT 10
                """

        if any(palabra in pregunta_lower for palabra in ['ciudad', 'sucursal', 'dónde', 'ubicación']):
            # Análisis por ubicación
            return f"""
            SELECT Sucursal,
                   COUNT(*) as num_ventas,
                   SUM(Precio_unitario * Cantidad) as total_ingresos
            FROM {self.table_name}
            GROUP BY Sucursal
            ORDER BY num_ventas DESC
            """

        if any(palabra in pregunta_lower for palabra in ['promedio', 'precio promedio', 'categoría']):
            # Análisis de precios por categoría
            return f"""
            SELECT Categoría,
                   COUNT(*) as num_productos,
                   AVG(Precio_unitario) as precio_promedio,
                   MIN(Precio_unitario) as precio_min,
                   MAX(Precio_unitario) as precio_max
            FROM {self.table_name}
            GROUP BY Categoría
            ORDER BY precio_promedio DESC
            """

        # Query general mejorada
        select_cols = ', '.join(cols_relevantes[:3]) if cols_relevantes else '*'
        return f"SELECT {select_cols} FROM {self.table_name} LIMIT 15"

    def ejecutar_consulta(self, sql):
        """Ejecutar SQL"""
        try:
            sql_limpio = ' '.join(line.strip() for line in sql.strip().split('\n') if line.strip())
            resultado = pd.read_sql_query(sql_limpio, self.conn)
            return sql_limpio, resultado
        except Exception as e:
            print(f"Error SQL: {e}")
            sql_simple = f"SELECT * FROM {self.table_name} LIMIT 10"
            resultado = pd.read_sql_query(sql_simple, self.conn)
            return sql_simple, resultado

    def responder(self, pregunta, usar_groq=True):
        """Responder pregunta completa"""
        try:
            # Generar y ejecutar SQL
            sql = self.generar_sql(pregunta)
            sql_ejecutado, datos = self.ejecutar_consulta(sql)

            print(f"SQL ejecutado: {sql_ejecutado}")
            print(f"Resultados: {len(datos)} filas\n")

            # Crear contexto
            contexto = f"""Pregunta: {pregunta}
SQL ejecutado: {sql_ejecutado}
Resultados: {len(datos)} filas encontradas

"""

            if not datos.empty:
                # Estadísticas automáticas
                cols_numericas = datos.select_dtypes(include=[np.number]).columns
                if len(cols_numericas) > 0:
                    contexto += "Estadísticas calculadas:\n"
                    for col in cols_numericas[:3]:
                        contexto += f"- {col}: total={datos[col].sum():.2f}, promedio={datos[col].mean():.2f}, máximo={datos[col].max():.2f}\n"
                    contexto += "\n"

                contexto += "Datos encontrados:\n"
                contexto += datos.to_string(index=False)

            # Respuesta con Groq si está disponible
            if usar_groq:
                try:
                    groq_key = userdata.get('GROQ_KEY')
                    client = Groq(api_key=groq_key)

                    response = client.chat.completions.create(
                        model="llama3-8b-8192",
                        messages=[{
                            "role": "user",
                            "content": f"""Analiza estos datos y responde de forma clara y específica:

{contexto}

Instrucciones:
- Responde basándote ÚNICAMENTE en los datos mostrados
- Da números exactos y estadísticas precisas
- Responde en español de forma profesional
- Si hay un ranking, muestra los top 3-5 elementos

Pregunta: {pregunta}"""
                        }],
                        max_tokens=400
                    )

                    return response.choices[0].message.content

                except Exception as e:
                    print(f"Error con Groq: {e}")
                    usar_groq = False

            # Respuesta automática si no hay Groq
            if not datos.empty:
                respuesta = f"Análisis para: {pregunta}\n\n"
                respuesta += f"Encontré {len(datos)} resultados:\n\n"
                respuesta += datos.head(10).to_string(index=False)

                if len(cols_numericas) > 0:
                    respuesta += f"\n\nEstadísticas principales:\n"
                    for col in cols_numericas[:2]:
                        respuesta += f"- {col}: Total = {datos[col].sum():.2f}, Promedio = {datos[col].mean():.2f}\n"

                return respuesta
            else:
                return "No se encontraron datos para esta consulta."

        except Exception as e:
            return f"Error procesando consulta: {e}"

# ===== FUNCIÓN DE USO SIMPLE =====
def iniciar_rag():
    """Inicializar RAG con el dataset"""
    rag = RAGSimple()
    rag.cargar_datos('https://raw.githubusercontent.com/CamiloVga/Curso-IA-Para-Ciencia-de-Datos/main/datos_tienda_ropa.csv')
    return rag

def preguntar(rag, pregunta):
    """Hacer una pregunta al RAG"""
    return rag.responder(pregunta, usar_groq=True)

# ===== CÓDIGO DE PRUEBA =====
if __name__ == "__main__":
    print("🚀 INICIANDO SISTEMA RAG")
    rag = iniciar_rag()

    # Preguntas de prueba
    preguntas_test = [
        "¿cuáles son los productos más vendidos?",
        "¿cuál es el método de pago más común?",
        "¿qué producto genera más ingresos?",
        "¿cuál es el precio promedio por categoría?"
    ]

    for i, pregunta in enumerate(preguntas_test, 1):
        print(f"\n{'='*60}")
        print(f"PREGUNTA {i}: {pregunta}")
        print('='*60)

        respuesta = preguntar(rag, pregunta)
        print(respuesta)

In [ ]:
# RAG BASE DE DATOS CON GROQ API (Base en directorio)
# Sistema de consultas en bases de datos usando embeddings y Groq para generación
# DIFERENCIAS vs RAG normal: embeddings de metadatos + generación automática de SQL

# Instalación de dependencias
!pip install groq sentence-transformers pandas numpy scikit-learn -q

import pandas as pd
import numpy as np
import sqlite3
import os
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from groq import Groq
from google.colab import userdata

# Configuración del sistema
GROQ_API_KEY = userdata.get('GROQ_KEY')
client = Groq(api_key=GROQ_API_KEY)
folder_path = '/content/carpeta_rag'

# Variables globales del sistema
model = SentenceTransformer('all-MiniLM-L6-v2')
conn = None
table_name = 'data'
column_embeddings = {}  # Embeddings de metadatos de columnas (no datos raw)
column_types = {}       # Tipos de datos para generar SQL inteligente

def load_database(folder_path):
    """Carga CSV desde carpeta y crea base de datos en memoria"""
    global conn, column_embeddings, column_types

    os.makedirs(folder_path, exist_ok=True)

    # Buscar archivo CSV en la carpeta
    csv_file = None
    for filename in os.listdir(folder_path):
        if filename.endswith('.csv'):
            csv_file = os.path.join(folder_path, filename)
            break

    if not csv_file:
        print("No se encontró archivo CSV en la carpeta.")
        return False

    print(f"Cargando {csv_file}...")
    df = pd.read_csv(csv_file)
    conn = sqlite3.connect(':memory:')
    df.to_sql(table_name, conn, index=False, if_exists='replace')

    # Analizar columnas y crear embeddings
    # CLAVE: No embebemos los datos, sino descripciones de las columnas
    # Esto permite mapear preguntas a columnas relevantes de la DB
    for col in df.columns:
        if df[col].dtype in ['int64', 'float64']:
            column_types[col] = 'numeric'
            desc = f"{col} valores numéricos entre {df[col].min()} y {df[col].max()}"
        elif df[col].nunique() <= 20:
            column_types[col] = 'categorical'
            valores = ', '.join(map(str, df[col].unique()[:5]))
            desc = f"{col} categorías como: {valores}"
        else:
            column_types[col] = 'text'
            desc = f"{col} texto libre"

        # Embedding de la descripción de la columna, no del contenido
        column_embeddings[col] = model.encode([desc])[0]

    print(f"Base de datos cargada: {df.shape[0]} filas, {df.shape[1]} columnas")
    return True

def find_relevant_columns(query, top_k=3):
    """Encuentra columnas más relevantes usando similitud coseno
    Diferencia clave vs RAG normal: buscamos columnas, no fragmentos de texto"""
    query_emb = model.encode([query])[0]
    scores = []

    for col, col_emb in column_embeddings.items():
        sim = cosine_similarity([query_emb], [col_emb])[0][0]
        scores.append((col, sim))

    return sorted(scores, key=lambda x: x[1], reverse=True)[:top_k]

def generate_sql(query):
    """Genera consulta SQL inteligente basada en la pregunta
    INNOVACIÓN: Combina embeddings + detección de patrones para SQL automático
    RAG tradicional solo busca texto, aquí generamos código ejecutable"""
    cols_relevantes = [col for col, _ in find_relevant_columns(query)]
    cols_numericas = [c for c in cols_relevantes if column_types.get(c) == 'numeric']
    cols_categoricas = [c for c in cols_relevantes if column_types.get(c) == 'categorical']

    query_lower = query.lower()

    # Detección de patrones en la pregunta
    if any(palabra in query_lower for palabra in ['más vendido', 'más popular', 'qué producto', 'cuáles productos']):
        if cols_numericas and cols_categoricas:
            return f"""
            SELECT {cols_categoricas[0]} as producto,
                   SUM({cols_numericas[0]}) as total_cantidad,
                   COUNT(*) as num_ventas
            FROM {table_name}
            GROUP BY {cols_categoricas[0]}
            ORDER BY total_cantidad DESC
            LIMIT 10
            """

    if any(palabra in query_lower for palabra in ['método pago', 'forma pago', 'cómo pagan', 'pago más común']):
        return f"""
        SELECT Método_pago,
               COUNT(*) as cantidad,
               ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM {table_name}), 1) as porcentaje
        FROM {table_name}
        GROUP BY Método_pago
        ORDER BY cantidad DESC
        """

    if any(palabra in query_lower for palabra in ['ingresos', 'revenue', 'dinero', 'ganancias']):
        if 'Precio_unitario' in column_types and 'Cantidad' in column_types:
            return f"""
            SELECT {cols_categoricas[0] if cols_categoricas else 'Producto'} as categoria,
                   SUM(Precio_unitario * Cantidad) as total_ingresos,
                   AVG(Precio_unitario * Cantidad) as ingreso_promedio
            FROM {table_name}
            GROUP BY {cols_categoricas[0] if cols_categoricas else 'Producto'}
            ORDER BY total_ingresos DESC
            LIMIT 10
            """

    if any(palabra in query_lower for palabra in ['ciudad', 'sucursal', 'dónde', 'ubicación']):
        return f"""
        SELECT Sucursal,
               COUNT(*) as num_ventas,
               SUM(Precio_unitario * Cantidad) as total_ingresos
        FROM {table_name}
        GROUP BY Sucursal
        ORDER BY num_ventas DESC
        """

    if any(palabra in query_lower for palabra in ['promedio', 'precio promedio', 'categoría']):
        return f"""
        SELECT Categoría,
               COUNT(*) as num_productos,
               AVG(Precio_unitario) as precio_promedio,
               MIN(Precio_unitario) as precio_min,
               MAX(Precio_unitario) as precio_max
        FROM {table_name}
        GROUP BY Categoría
        ORDER BY precio_promedio DESC
        """

    # Query general
    select_cols = ', '.join(cols_relevantes[:3]) if cols_relevantes else '*'
    return f"SELECT {select_cols} FROM {table_name} LIMIT 15"

def execute_query(sql):
    """Ejecuta consulta SQL y retorna resultados
    Diferencia vs RAG normal: ejecutamos código contra DB real, no solo texto"""
    try:
        sql_clean = ' '.join(line.strip() for line in sql.strip().split('\n') if line.strip())
        resultado = pd.read_sql_query(sql_clean, conn)
        return sql_clean, resultado
    except Exception as e:
        print(f"Error SQL: {e}")
        sql_simple = f"SELECT * FROM {table_name} LIMIT 10"
        resultado = pd.read_sql_query(sql_simple, conn)
        return sql_simple, resultado

def generate_answer(query, sql_executed, data):
    """Genera respuesta usando Groq API con los datos encontrados
    Contexto especial: incluye SQL ejecutado + estadísticas calculadas automáticamente"""
    if data.empty:
        return "No se encontraron datos para esta consulta."

    # Crear contexto con estadísticas automáticas
    context = f"""Pregunta: {query}
SQL ejecutado: {sql_executed}
Resultados: {len(data)} filas encontradas

"""

    # Agregar estadísticas de columnas numéricas
    cols_numericas = data.select_dtypes(include=[np.number]).columns
    if len(cols_numericas) > 0:
        context += "Estadísticas calculadas:\n"
        for col in cols_numericas[:3]:
            context += f"- {col}: total={data[col].sum():.2f}, promedio={data[col].mean():.2f}, máximo={data[col].max():.2f}\n"
        context += "\n"

    context += "Datos encontrados:\n"
    context += data.to_string(index=False)

    # Generar respuesta con Groq
    prompt = f"""Analiza estos datos y responde de forma clara y específica:

{context}

Instrucciones:
- Responde basándote ÚNICAMENTE en los datos mostrados
- Da números exactos y estadísticas precisas
- Responde en español de forma profesional
- Si hay un ranking, muestra los top 3-5 elementos

Pregunta: {query}"""

    try:
        response = client.chat.completions.create(
            model="llama3-8b-8192",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=400
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error generando respuesta: {e}"

def inicializar_rag():
    """Inicializa el sistema RAG cargando base de datos"""
    print("Iniciando sistema RAG...")
    if load_database(folder_path):
        print("Sistema RAG listo para consultas.")
        return True
    return False

# Inicialización del sistema
inicializar_rag()

# BLOQUE DE INFERENCIA - Ejecutar por separado
query = "¿cuáles son los productos más vendidos?"  # Cambia tu pregunta aquí
sql = generate_sql(query)
sql_executed, data = execute_query(sql)
respuesta = generate_answer(query, sql_executed, data)
print(f"Pregunta: {query}")
print(f"SQL: {sql_executed}")
print(f"Respuesta: {respuesta}")